#ML Framework : Retrieval Augmented Generation
#Name: Manu Sri Mogalapalli
#PRN: 1302250053
#Class: S Y BCom FinTech | AI ML 1 - CCA2

In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters

In [ ]:
college_rules = """
A project report with more than 20% plagiarism receives zero marks for that report.

Scholarship holders must maintain a CGPA of at least 7.5 to renew the scholarship every year.

The campus Wi-Fi password changes every Monday and must not be shared with outsiders.

Students can apply for the semester exchange program only after completing four semesters.

Permission for any college fest must be taken from the dean at least 10 days in advance.

A student who fails a subject can clear the backlog in a maximum of two attempts.

Two-wheelers need a parking sticker, which costs Rs 300 per year.

Grievances must be submitted on the online portal and are answered within 5 working days.

Safety goggles and lab coats are compulsory in the chemistry lab during practical sessions.

Student presentations must be delivered in formal business attire.

A gap year is allowed only with a written request and the registrar's approval.

Alumni may use the library for up to 3 days per month with a guest pass.
"""

In [ ]:
from langchain_core.documents import Document

# One rule = one chunk, so an answer never mixes two different rules
documents = [Document(page_content=r.strip()) for r in college_rules.strip().split("\n\n") if r.strip()]
print(len(documents), "rules loaded")

In [ ]:
import re

STOP = {"what","when","where","which","who","whom","how","does","do","is","are","can","could","should","would","will",
        "the","a","an","to","of","in","on","at","for","and","or","my","me","i","we","our","you","your","there","any","much",
        "many","need","needed","get","have","has","about","college","student","students",
        "allow","allowed","long","wait","reply","happens","happen","give","use","must"}

def keywords_from(question):
    """Turn a free-text question into search stems (first 6 letters, so 'plagiarism' matches 'plagiarising')."""
    words = re.findall(r"[a-zA-Z\-]+", question.lower())
    return [w[:6] for w in words if w not in STOP and len(w) > 2]

def ask(question, keywords=None, show_all=False):
    stems = [k.lower()[:6] for k in keywords] if keywords else keywords_from(question)
    scored = []
    for doc in documents:
        text = doc.page_content.lower()
        hits = sum(s in text for s in stems)
        if hits:
            scored.append((hits, doc.page_content))
    print("Q:", question)
    if not scored:
        print("A: No matching rule found.\n"); return
    scored.sort(key=lambda x: -x[0])
    best = scored[0][0]
    for hits, rule in scored:
        if show_all or hits == best:
            print("A:", rule)
    print()

In [ ]:
ask("What happens if my report has plagiarism?")
ask("What CGPA do I need to keep my scholarship?")
ask("Who can join the exchange program and when?")
ask("How many attempts do I get to clear a backlog?")
ask("How much does a parking sticker cost?")
ask("What should I wear for a presentation?")
ask("Can alumni use the library?")
ask("Do I need goggles in the lab?")
ask("How long do I have to wait for a reply to a complaint?")   # 'complaint' is not in the rule text -> correctly finds nothing
ask("How long do I have to wait for a reply to a grievance?")   # works: uses the rule's own word

### Ask your own questions


In [ ]:
while True:
    q = input("Your question (or 'quit'): ").strip()
    if q.lower() in {"quit", "exit", ""}:
        print("Bye!"); break
    ask(q)